# Notebook 3 — ML Model

XGBoost with SHAP explainability + walk-forward backtesting.

**Features**: all fraud signals + value metrics from each annual 10-K filing  
**Labels**: two targets trained separately:
- `forward_return_12m` — regression (predict exact return)
- `beat_sp500` — binary classification (did the stock beat S&P 500?)

**Validation strategy**: walk-forward splits — never train on future data:
```
Split 1: train 2019-2020 → test 2021
Split 2: train 2019-2021 → test 2022
Split 3: train 2019-2022 → test 2023
```

## Install dependencies (first time only)

In [ ]:
# Uncomment to install
# !pip install xgboost shap scikit-learn

## Load data

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

import xgboost as xgb
import shap
from sklearn.metrics import roc_auc_score, accuracy_score, mean_absolute_error
from sklearn.preprocessing import RobustScaler

# ── Load dataset ──────────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    import subprocess
    subprocess.run(['wget', '-q', '-O', 'historical_dataset.parquet',
        'https://raw.githubusercontent.com/sherlock718/stock-fraud-screener/main/data/historical_dataset.parquet'])
    path = 'historical_dataset.parquet'
else:
    path = '../data/historical_dataset.parquet'

df = pd.read_parquet(path)
print(f"Rows: {len(df):,} | Companies: {df['ticker'].nunique():,}")
print(f"Forward returns: {df['forward_return_12m'].notna().sum():,}")

## Feature engineering

In [ ]:
# All features used in the model
FEATURES = [
    # Fraud signals
    'beneish_score', 'piotroski_score', 'accruals_ratio', 'cfd_ratio',
    'altman_score', 'ar_ratio', 'dso', 'non_op_ratio',
    # Binary fraud flags (0/1)
    'beneish_flag', 'piotroski_weak', 'accruals_flag', 'cfd_flag',
    'altman_flag', 'revenue_quality_flag', 'earnings_quality_flag',
    # Value metrics
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'croic', 'roe', 'roa', 'fcf_yield',
    'gross_margin', 'net_margin', 'debt_to_equity', 'current_ratio',
    'pe_ratio', 'pb_ratio', 'ev_ebitda', 'ncav_ratio',
]

# Keep only rows with a forward return label
ml_df = df[df['forward_return_12m'].notna()].copy()

# Winsorise returns (clip extreme outliers that would dominate the loss)
ml_df['forward_return_12m'] = ml_df['forward_return_12m'].clip(
    ml_df['forward_return_12m'].quantile(0.01),
    ml_df['forward_return_12m'].quantile(0.99)
)

# Binary label: beat S&P 500
ml_df['beat_sp500'] = ml_df['beat_sp500'].astype(float)

# Only use features that exist
features = [f for f in FEATURES if f in ml_df.columns]
print(f"Features used: {len(features)}")

X = ml_df[features].copy()

# Winsorise each feature at 1/99th percentile
for col in X.select_dtypes(include=[float, int]).columns:
    lo, hi = X[col].quantile(0.01), X[col].quantile(0.99)
    X[col] = X[col].clip(lo, hi)

y_reg  = ml_df['forward_return_12m']
y_cls  = ml_df['beat_sp500']
years  = ml_df['fiscal_year']

print(f"Dataset: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Label balance (beat S&P): {y_cls.mean():.1%}")

## Walk-forward validation

Train on all years up to year T, test on year T+1. This strictly avoids look-ahead bias.

In [ ]:
test_years = sorted(years.unique())[2:]  # need at least 2 years to train
print(f"Walk-forward test years: {test_years}")

all_preds_cls  = []
all_preds_reg  = []
all_true_cls   = []
all_true_reg   = []
all_test_years = []
feature_importances = []

for test_year in test_years:
    train_mask = years < test_year
    test_mask  = years == test_year

    X_train, X_test = X[train_mask], X[test_mask]
    y_train_cls, y_test_cls = y_cls[train_mask], y_cls[test_mask]
    y_train_reg, y_test_reg = y_reg[train_mask], y_reg[test_mask]

    if len(X_train) < 50 or len(X_test) < 10:
        continue

    # ── Classification: beat S&P 500 ─────────────────────────────────────────
    clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', use_label_encoder=False,
        random_state=42, n_jobs=-1
    )
    clf.fit(X_train.fillna(X_train.median()), y_train_cls.fillna(0))
    pred_proba = clf.predict_proba(X_test.fillna(X_train.median()))[:, 1]
    all_preds_cls.extend(pred_proba)
    all_true_cls.extend(y_test_cls.values)

    # ── Regression: predict exact return ─────────────────────────────────────
    reg = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1
    )
    reg.fit(X_train.fillna(X_train.median()), y_train_reg.fillna(0))
    pred_reg = reg.predict(X_test.fillna(X_train.median()))
    all_preds_reg.extend(pred_reg)
    all_true_reg.extend(y_test_reg.values)
    all_test_years.extend([test_year] * len(y_test_cls))

    fi = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=False)
    feature_importances.append(fi)

    auc = roc_auc_score(y_test_cls.fillna(0), pred_proba)
    mae = mean_absolute_error(y_test_reg, pred_reg)
    print(f"  {test_year}: train={train_mask.sum():,} test={test_mask.sum():,} | AUC={auc:.3f} | MAE={mae:.3f}")

print(f"\nOverall AUC: {roc_auc_score(all_true_cls, all_preds_cls):.3f}")
print(f"Overall MAE: {mean_absolute_error(all_true_reg, all_preds_reg):.3f}")

## Feature importance (averaged across all walk-forward folds)

In [ ]:
avg_importance = pd.concat(feature_importances, axis=1).mean(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top20 = avg_importance.head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color='#3498db', alpha=0.85)
ax.set_title('Top 20 Features — Average Importance Across Walk-Forward Folds')
ax.set_xlabel('XGBoost Feature Importance (gain)')
plt.tight_layout()
plt.show()

print(avg_importance.head(20).to_string())

## SHAP — explainable predictions

SHAP shows HOW each feature pushes each prediction up or down.

In [ ]:
# Train final model on all data (for SHAP visualization only — not for live predictions)
X_all = X.fillna(X.median())
y_all = y_cls.fillna(0)

final_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', use_label_encoder=False,
    random_state=42, n_jobs=-1
)
final_clf.fit(X_all, y_all)

explainer = shap.TreeExplainer(final_clf)
shap_values = explainer.shap_values(X_all)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_all, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Beat S&P 500 Model')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm — shows direction of each feature's effect
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_all, show=False)
plt.title('SHAP Beeswarm — How Features Push Predictions')
plt.tight_layout()
plt.show()

## Backtest equity curve

Strategy: each year, go long the top-N stocks by predicted probability of beating S&P 500.
Compare cumulative returns vs holding SPY.

In [ ]:
TOP_N = 20  # hold top 20 stocks per year

backtest_df = ml_df.copy()
backtest_df = backtest_df[backtest_df['forward_return_12m'].notna()].copy()
backtest_df['pred_proba'] = 0.5  # placeholder — replace with actual predictions below

# Rebuild predictions using walk-forward (same splits as above)
pred_series = pd.Series(np.nan, index=backtest_df.index)
for test_year in test_years:
    train_mask = years < test_year
    test_mask  = years == test_year
    if train_mask.sum() < 50 or test_mask.sum() < 10:
        continue
    clf2 = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', use_label_encoder=False,
        random_state=42, n_jobs=-1
    )
    clf2.fit(X[train_mask].fillna(X[train_mask].median()), y_cls[train_mask].fillna(0))
    preds = clf2.predict_proba(X[test_mask].fillna(X[train_mask].median()))[:, 1]
    pred_series.loc[X[test_mask].index] = preds

backtest_df['pred_proba'] = pred_series
backtest_df = backtest_df[backtest_df['pred_proba'].notna()]

# For each year, select top N by predicted probability
results = []
for yr, grp in backtest_df.groupby('fiscal_year'):
    top = grp.nlargest(TOP_N, 'pred_proba')
    strategy_ret  = top['forward_return_12m'].mean()
    sp500_ret     = top['sp500_return_12m'].mean() if 'sp500_return_12m' in top.columns else None
    results.append({'year': yr, 'strategy': strategy_ret, 'sp500': sp500_ret,
                    'n_stocks': len(top)})

res_df = pd.DataFrame(results).dropna(subset=['strategy'])

# Cumulative return
res_df['cum_strategy'] = (1 + res_df['strategy']).cumprod()
res_df['cum_sp500']    = (1 + res_df['sp500'].fillna(0)).cumprod()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual returns
x = np.arange(len(res_df))
w = 0.35
axes[0].bar(x - w/2, res_df['strategy'], w, label='Strategy (Top 20)', color='#2ecc71', alpha=0.85)
axes[0].bar(x + w/2, res_df['sp500'],    w, label='S&P 500',           color='#3498db', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(res_df['year'].astype(int))
axes[0].axhline(0, color='white', alpha=0.3)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_title(f'Annual Returns — Strategy vs S&P 500')
axes[0].legend()

# Cumulative
axes[1].plot(res_df['year'], res_df['cum_strategy'], marker='o', color='#2ecc71', label='Strategy')
axes[1].plot(res_df['year'], res_df['cum_sp500'],    marker='o', color='#3498db', label='S&P 500')
axes[1].set_title('Cumulative Return')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
axes[1].legend()

plt.suptitle(f'Backtest: Top {TOP_N} stocks by ML score each year', y=1.02)
plt.tight_layout()
plt.show()

print(res_df[['year', 'strategy', 'sp500', 'n_stocks']].to_string(index=False))
print(f"\nFinal strategy return: {res_df['cum_strategy'].iloc[-1]-1:.1%}")
print(f"Final S&P 500 return:  {res_df['cum_sp500'].iloc[-1]-1:.1%}")

## Inspect individual predictions

For any company, show SHAP waterfall — exactly why the model gave that prediction.

In [ ]:
# Change this to inspect any ticker
INSPECT_TICKER = 'AAPL'

rows = X_all[ml_df['ticker'] == INSPECT_TICKER]
if len(rows) == 0:
    print(f"No data found for {INSPECT_TICKER}")
else:
    # Show most recent year
    row_idx = rows.index[-1]
    row = rows.loc[[row_idx]]
    year = ml_df.loc[row_idx, 'fiscal_year']
    actual = ml_df.loc[row_idx, 'forward_return_12m']
    print(f"{INSPECT_TICKER} — FY{year} | Actual 12m return: {actual:.1%}")

    shap_row = explainer.shap_values(row)
    shap.waterfall_plot(
        shap.Explanation(values=shap_row[0],
                         base_values=explainer.expected_value,
                         data=row.values[0],
                         feature_names=features),
        show=True
    )